# Stage 2 — Inference Acceleration
## Notebook 2: vLLM Deployment

**Objective:** Install, configure, and deploy vLLM for high-performance LLM serving. Learn key parameters, build concurrent clients, explore monitoring, and handle graceful shutdown.

**Prerequisites:** Familiarity with LLM inference concepts, basic HTTP/API knowledge.

In [ ]:
# ============================
# Cell 1: Install dependencies
# ============================
!pip install -q openai httpx aiohttp asyncio
!pip install -q numpy matplotlib
!pip install -q vllm  # Requires CUDA GPU; skip on CPU-only machines
!pip install -q prometheus_client

print("Dependencies installed.")
print("Note: vLLM requires CUDA-capable GPU for actual inference.")
print("Client-side code (OpenAI API calls) works on any machine.")

## 1. What is vLLM?

vLLM is a high-throughput, memory-efficient LLM serving engine. It powers many production deployments thanks to three key innovations:

- **PagedAttention**: Manages KV cache like OS virtual memory, eliminating ~60% memory waste
- **Continuous batching**: Dynamically adds/removes requests from an ongoing batch -- no idle GPU time
- **OpenAI-compatible API**: Drop-in replacement for applications using OpenAI SDK

**Performance claims:**
- 14-24x higher throughput than vanilla Hugging Face Transformers
- 2-3x higher throughput than Hugging Face TGI
- Supports quantization (GPTQ, AWQ, SqueezeLLM, FP8)
- Tensor parallelism for multi-GPU setups

### Architecture Overview

```
+-------------+     +--------------+     +-------------------+
|   Client    |---->|  API Server  |---->|  Scheduler        |
| (OpenAI SDK)|     |  (FastAPI)   |     |  (Continuous      |
|             |<----|              |<----|   Batching +       |
+-------------+     +--------------+     |   PagedAttention) |
                                          +--------+----------+
                                                   |
                                          +--------v----------+
                                          |  GPU Executor      |
                                          |  - KV Cache Blocks |
                                          |  - Model Weights   |
                                          +-------------------+
```

## 2. Launching a vLLM Server

vLLM provides two ways to serve models: **CLI** (for production) and **Python API** (for embedding in applications). Both produce an OpenAI-compatible endpoint at `http://localhost:8000/v1`.

In [ ]:
# ===========================================
# Cell 3: vLLM CLI -- Production Launch Command
# ===========================================
# NOTE: GPU REQUIRED -- this is reference, not runnable in this notebook.

print("=" * 70)
print("METHOD 1: CLI Launch (run in terminal on GPU machine)")
print("=" * 70)
print("""
# ---- Basic launch ----
python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-0.5B \\
    --host 0.0.0.0 \\
    --port 8000

# ---- Production launch with all key parameters ----
python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-2-7b-chat-hf \\
    --host 0.0.0.0 \\
    --port 8000 \\
    --tensor-parallel-size 1 \\
    --max-model-len 4096 \\
    --max-num-batched-tokens 8192 \\
    --max-num-seqs 256 \\
    --gpu-memory-utilization 0.90 \\
    --enable-prefix-caching \\
    --swap-space 4 \\
    --dtype auto
""")

In [ ]:
# =============================================
# Cell 4: Python API -- Start vLLM programmatically
# =============================================
# NOTE: GPU REQUIRED

print("=" * 70)
print("METHOD 2: Python API Launch")
print("=" * 70)
print("""
import subprocess, sys

def start_vllm_server(model="Qwen/Qwen2.5-0.5B", host="0.0.0.0", port=8000, **kwargs):
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model, "--host", host, "--port", str(port)]
    for k, v in kwargs.items():
        cmd.extend([f"--{k.replace('_', '-')}", str(v)])
    return subprocess.Popen(cmd)

server = start_vllm_server(model="Qwen/Qwen2.5-0.5B",
                           max_model_len=2048, gpu_memory_utilization=0.85,
                           enable_prefix_caching=True)
""")

In [ ]:
# =============================================
# Cell 5: vLLM as a library (offline inference)
# =============================================
# NOTE: GPU REQUIRED

print("=" * 70)
print("METHOD 3: vLLM as a Library (Embedded Inference)")
print("=" * 70)
print("""
from vllm import LLM, SamplingParams

llm = LLM(model="Qwen/Qwen2.5-0.5B", tensor_parallel_size=1,
          max_model_len=2048, gpu_memory_utilization=0.85,
          trust_remote_code=True)

sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=256)

prompts = ["Explain quantum computing:", "Write a haiku about AI:"]
outputs = llm.generate(prompts, sampling_params)

for prompt, output in zip(prompts, outputs):
    print(f"Prompt: {prompt}")
    print(f"Output: {output.outputs[0].text}")
""")
print("Library mode is excellent for batch processing and offline evaluation.")

## 3. Key Parameters Deep Dive

Understanding vLLM's scheduling parameters is critical for performance tuning. The three most important knobs are `max-num-batched-tokens`, `max-num-seqs`, and `gpu-memory-utilization`.

In [ ]:
# ============================================
# Cell 7: Key vLLM Parameters Reference
# ============================================

params = [
    {
        "name": "max-num-batched-tokens", "default": "8192",
        "description": "Max tokens in one forward pass (prompt + generation across all concurrent seqs).",
        "tuning": "For A100 80GB with 7B model, try 16384-32768.",
        "too_high": "OOM errors, high scheduling overhead",
        "too_low": "GPU underutilization, requests queued"
    },
    {
        "name": "max-num-seqs", "default": "256",
        "description": "Maximum concurrent sequences processed at once.",
        "tuning": "Limited by KV cache memory. 7B on A100: ~128-256 typical.",
        "too_high": "KV cache exhausted, requests preempted to CPU",
        "too_low": "Throughput limited for parallel workloads"
    },
    {
        "name": "gpu-memory-utilization", "default": "0.90",
        "description": "Fraction of GPU memory for model + KV cache.",
        "tuning": "0.85-0.92 safe. Leave ~300-500 MB for CUDA context.",
        "too_high": "CUDA OOM during loading or first inference",
        "too_low": "Wasted GPU memory, fewer concurrent requests"
    },
    {
        "name": "enable-prefix-caching", "default": "False",
        "description": "Cache KV blocks for identical prefixes (system prompts) across requests.",
        "tuning": "Enable for chatbots/RAG. Disable if all prompts unique.",
        "too_high": "Slightly more memory for block table metadata",
        "too_low": "Redundant computation for shared prefixes"
    },
    {
        "name": "swap-space", "default": "4 GB",
        "description": "CPU swap space for preempted KV cache blocks.",
        "tuning": "4 GB enough for most. 16+ for long contexts.",
        "too_high": "Latency penalty on swap",
        "too_low": "Requests rejected under load"
    },
    {
        "name": "max-model-len", "default": "Model max",
        "description": "Maximum sequence length (prompt + completion).",
        "tuning": "Lower = more KV cache blocks for other seqs.",
        "too_high": "Each seq reserves more blocks, reducing concurrency",
        "too_low": "Long prompts rejected"
    },
]

for p in params:
    print(f"\n{'=' * 65}")
    print(f"  --{p['name']}  (default: {p['default']})")
    print(f"{'=' * 65}")
    print(f"  Description:  {p['description']}")
    print(f"  Tuning:       {p['tuning']}")
    print(f"  Too high:     {p['too_high']}")
    print(f"  Too low:      {p['too_low']}")

## 4. OpenAI-Compatible Client

vLLM's API is fully compatible with the OpenAI client library. Just change `base_url` to point to your vLLM server. This means you can swap between OpenAI and vLLM with zero code changes.

In [ ]:
# ============================================
# Cell 9: OpenAI-Compatible Client -- Non-streaming
# ============================================
# Works against any running vLLM server.

from openai import OpenAI

VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_API_KEY = "not-needed"  # vLLM skips auth by default

client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)

print("=== Non-Streaming Completion ===\n")
try:
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-0.5B",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain what vLLM is in one paragraph."}
        ],
        max_tokens=256, temperature=0.7, top_p=0.95,
    )
    print(f"Response ID: {response.id}")
    print(f"Model: {response.model}")
    print(f"Tokens: total={response.usage.total_tokens} (prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens})")
    print(f"Content:\n{response.choices[0].message.content}")
except Exception as e:
    print(f"[EXPECTED] vLLM server not reachable: {e}")
    print("Start a vLLM server on port 8000 to run this section.")

In [ ]:
# ============================================
# Cell 10: OpenAI-Compatible Client -- Streaming
# ============================================

print("=== Streaming Completion ===\n")
try:
    stream = client.chat.completions.create(
        model="Qwen/Qwen2.5-0.5B",
        messages=[{"role": "user", "content": "Write a Python function to calculate Fibonacci numbers."}],
        max_tokens=256, temperature=0.3, stream=True,
    )
    full_text = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            token_text = chunk.choices[0].delta.content
            print(token_text, end="", flush=True)
            full_text += token_text
    print(f"\n\n[Received {len(full_text)} characters]")
except Exception as e:
    print(f"[EXPECTED] vLLM server not reachable: {e}")

In [ ]:
# ============================================
# Cell 11: List available models + health check
# ============================================

try:
    models_response = client.models.list()
    print(f"=== Available Models ({len(models_response.data)}) ===")
    for model in models_response.data:
        print(f"  - {model.id} (owned by: {model.owned_by})")

    import httpx
    health = httpx.get("http://localhost:8000/health", timeout=5.0)
    print(f"\nHealth check: HTTP {health.status_code}")
except Exception as e:
    print(f"[EXPECTED] vLLM server not reachable: {e}")

## 5. Concurrent Requests with asyncio

This is one of vLLM's killer features -- handling many requests simultaneously with continuous batching. We use `asyncio.gather` to fire off multiple requests and measure throughput.

In [ ]:
# ==================================================
# Cell 13: Concurrent Request Testing with asyncio.gather
# ==================================================

import asyncio
import time
from openai import AsyncOpenAI

async_client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)

async def send_request_async(request_id, semaphore):
    async with semaphore:
        start = time.perf_counter()
        try:
            response = await async_client.chat.completions.create(
                model="Qwen/Qwen2.5-0.5B",
                messages=[{"role": "user", "content": f"[Request {request_id}] Write a short poem about AI."}],
                max_tokens=64, temperature=0.7,
            )
            elapsed = time.perf_counter() - start
            return {"id": request_id, "success": True, "latency": elapsed,
                    "tokens": response.usage.completion_tokens,
                    "tok_per_sec": response.usage.completion_tokens / elapsed}
        except Exception as e:
            return {"id": request_id, "success": False, "latency": time.perf_counter() - start, "error": str(e)}

async def run_concurrent_benchmark(num_requests=16, max_concurrent=8):
    semaphore = asyncio.Semaphore(max_concurrent)
    print(f"Sending {num_requests} requests (max {max_concurrent} concurrent)...")
    overall_start = time.perf_counter()
    tasks = [send_request_async(i, semaphore) for i in range(num_requests)]
    results = await asyncio.gather(*tasks)
    overall_elapsed = time.perf_counter() - overall_start

    successful = [r for r in results if r["success"]]
    failed = [r for r in results if not r["success"]]
    print(f"\n{'=' * 60}")
    print(f"BENCHMARK RESULTS")
    print(f"{'=' * 60}")
    print(f"Total requests: {num_requests} | Successful: {len(successful)} | Failed: {len(failed)}")
    print(f"Total time: {overall_elapsed:.2f}s")

    if successful:
        latencies = sorted([r["latency"] for r in successful])
        tokens = [r["tokens"] for r in successful]
        print(f"\nLatency -- P50: {latencies[len(latencies)//2]:.3f}s | P95: {latencies[int(len(latencies)*0.95)]:.3f}s | Max: {latencies[-1]:.3f}s")
        total_tokens = sum(tokens)
        print(f"Throughput -- {total_tokens/overall_elapsed:.1f} tok/s | {len(successful)/overall_elapsed:.1f} req/s")
    return results

print("Attempting concurrent benchmark...")
print("(Start a vLLM server first to see real results)\n")
try:
    results = await run_concurrent_benchmark(num_requests=8, max_concurrent=4)
except Exception as e:
    print(f"[EXPECTED] Server not available: {e}")
    print("To run: python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-0.5B")

## 6. Performance Tuning Walkthrough

Understanding how `max-num-seqs` affects the throughput-latency tradeoff is essential for production deployment.

In [ ]:
# =============================================
# Cell 15: Performance Tuning Visualization
# =============================================

import matplotlib.pyplot as plt
import numpy as np

max_seqs_values = [4, 8, 16, 32, 64, 128, 256]
throughput_tok_s = [450, 820, 1450, 2200, 2800, 3100, 3200]
p50_latency_s = [0.8, 1.1, 1.6, 2.4, 3.8, 6.2, 11.5]
p99_latency_s = [1.2, 1.8, 2.9, 4.5, 8.1, 15.3, 28.7]
gpu_util_pct = [42, 58, 72, 85, 92, 96, 97]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(max_seqs_values, throughput_tok_s, 'o-', linewidth=2, markersize=8, color='green')
axes[0].set_xlabel('max-num-seqs'); axes[0].set_ylabel('Throughput (tok/s)')
axes[0].set_title('Throughput vs Max Sequences'); axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log', base=2)

axes[1].plot(max_seqs_values, p50_latency_s, 's-', linewidth=2, markersize=8, label='P50', color='blue')
axes[1].plot(max_seqs_values, p99_latency_s, '^-', linewidth=2, markersize=8, label='P99', color='red')
axes[1].set_xlabel('max-num-seqs'); axes[1].set_ylabel('Latency (seconds)')
axes[1].set_title('Latency vs Max Sequences (tradeoff!)'); axes[1].grid(True, alpha=0.3)
axes[1].legend(); axes[1].set_xscale('log', base=2)

axes[2].plot(max_seqs_values, gpu_util_pct, 'D-', linewidth=2, markersize=8, color='purple')
axes[2].set_xlabel('max-num-seqs'); axes[2].set_ylabel('GPU Utilization (%)')
axes[2].set_title('GPU Utilization'); axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log', base=2)

plt.tight_layout(); plt.show()

print("=== Tuning Guidelines ===\n")
print("1. Throughput-optimized (batch): --max-num-seqs 128-256, tolerate higher P99")
print("2. Latency-sensitive (chat):    --max-num-seqs 16-32, accept lower throughput")
print("3. Balanced (API serving):      --max-num-seqs 64, sweet spot")

In [ ]:
# ============================================
# Cell 16: Continuous vs Static Batching Visual
# ============================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

static_timeline = []
for batch_idx in range(5):
    for req in range(4):
        start = batch_idx * 2.0
        static_timeline.append((start, start + 1.8, req + batch_idx * 4))
for start, end, req_id in static_timeline:
    ax1.barh(req_id, end - start, left=start, height=0.6, color='steelblue', alpha=0.7)
ax1.set_xlabel('Time (seconds)'); ax1.set_ylabel('Request ID')
ax1.set_title('Static Batching\n(requests wait for batch to fill)'); ax1.set_xlim(0, 12)

cont_timeline = [(0.0, 1.2, 0), (0.1, 1.3, 1), (0.3, 1.5, 2), (0.5, 1.7, 3),
                 (1.2, 2.4, 4), (1.3, 2.5, 5), (1.5, 2.7, 6), (2.4, 3.6, 7), (2.5, 3.7, 8)]
for start, end, req_id in cont_timeline:
    ax2.barh(req_id, end - start, left=start, height=0.6, color='coral', alpha=0.7)
ax2.set_xlabel('Time (seconds)'); ax2.set_ylabel('Request ID')
ax2.set_title('Continuous Batching (vLLM)\n(requests added/removed dynamically)'); ax2.set_xlim(0, 5)

plt.tight_layout(); plt.show()
print("Continuous batching eliminates idle time -- new requests join as others finish.")

## 7. Metrics Endpoint Exploration

vLLM exposes a Prometheus-compatible `/metrics` endpoint with detailed performance metrics: request counts, KV cache usage, TTFT, and TPOT.

In [ ]:
# ==============================================
# Cell 18: vLLM Prometheus Metrics Reference
# ==============================================

print("=" * 65)
print("vLLM Prometheus Metrics Endpoint")
print("=" * 65)
print("\nWhen vLLM is running, visit: http://localhost:8000/metrics\n")

metrics_doc = [
    ("vllm:num_requests_running", "Requests currently being processed"),
    ("vllm:num_requests_waiting", "Requests in the waiting queue"),
    ("vllm:num_requests_swapped", "Requests swapped to CPU memory"),
    ("vllm:gpu_cache_usage_perc", "GPU KV cache blocks in use (%)"),
    ("vllm:cpu_cache_usage_perc", "CPU swap blocks in use (%)"),
    ("vllm:time_to_first_token_seconds", "Histogram of TTFT"),
    ("vllm:time_per_output_token_seconds", "Histogram of TPOT"),
    ("vllm:request_prompt_tokens", "Histogram of prompt token counts"),
    ("vllm:request_generation_tokens", "Histogram of generation token counts"),
    ("vllm:request_success", "Counter of successful requests"),
    ("vllm:e2e_request_latency_seconds", "Histogram of end-to-end latency"),
    ("vllm:iteration_tokens_total", "Counter: tokens per iteration"),
]
for metric, desc in metrics_doc:
    print(f"  {metric}")
    print(f"    -> {desc}\n")

print("Prometheus scrape config:")
print("""
scrape_configs:
  - job_name: 'vllm'
    static_configs:
      - targets: ['localhost:8000']
    scrape_interval: 15s
""")

In [ ]:
# ============================================
# Cell 19: Fetch and Parse Metrics
# ============================================

import re

def parse_metrics(metrics_text):
    """Parse Prometheus text format metrics into a dict."""
    metrics = {}
    for line in metrics_text.strip().split('\n'):
        if line.startswith('#') or not line.strip():
            continue
        match = re.match(r'(\w+)(?:\{([^}]+)\})?\s+([\d.e+-]+)', line)
        if match:
            name, labels, value = match.groups()
            key = f"{name}{{{labels}}}" if labels else name
            try:
                metrics[key] = float(value)
            except ValueError:
                pass
    return metrics

try:
    import httpx
    response = httpx.get("http://localhost:8000/metrics", timeout=5.0)
    if response.status_code == 200:
        metrics = parse_metrics(response.text)
        print("Current vLLM Metrics:")
        for prefix in ['vllm:num_requests_running', 'vllm:num_requests_waiting',
                       'vllm:gpu_cache_usage_perc', 'vllm:request_success']:
            for key, value in metrics.items():
                if key.startswith(prefix):
                    print(f"  {key}: {value}")
    else:
        print(f"Metrics endpoint returned status {response.status_code}")
except Exception as e:
    print(f"[EXPECTED] Cannot reach metrics endpoint: {e}")

## 8. Multi-Model Serving

vLLM supports serving multiple models. For production, running separate vLLM instances per model (with a routing layer like LiteLLM) is the recommended approach.

In [ ]:
# ============================================
# Cell 21: Multi-Model Serving Strategies
# ============================================

print("vLLM Multi-Model Serving Strategies\n" + "=" * 70 + "\n")

print("Strategy 1: Single vLLM with LoRA adapters")
print("-" * 45)
print("""
python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-2-7b-hf \\
    --enable-lora --max-loras 4 --max-lora-rank 64 \\
    --lora-modules my-adapter=./path/to/lora/
""")

print("Strategy 2: Multiple vLLM instances (RECOMMENDED)")
print("-" * 45)
print("""
# Fast model on port 8000
python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-0.5B --port 8000

# Powerful model on port 8001
python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-72B --port 8001 --tensor-parallel-size 4

# Router: litellm --model fast=localhost:8000 --model powerful=localhost:8001
""")

print("Strategy 3: API Gateway with LiteLLM")
print("-" * 45)
print("""
# LiteLLM proxy config.yaml:
model_list:
  - model_name: gpt-3.5-turbo
    litellm_params:
      model: openai/Qwen2.5-7B
      api_base: http://localhost:8000/v1
      api_key: none
  - model_name: gpt-4
    litellm_params:
      model: openai/Qwen2.5-72B
      api_base: http://localhost:8001/v1
      api_key: none
""")

## 9. Error Handling and Graceful Shutdown

Production systems need robust error handling (exponential backoff for retries) and graceful shutdown (drain in-flight requests before terminating).

In [ ]:
# =============================================
# Cell 23: Client-Side Error Handling with Retry
# =============================================

import time as time_mod
from openai import APIError, APIConnectionError, RateLimitError, APITimeoutError

def robust_chat_completion(client, messages, max_retries=3, base_delay=1.0, **kwargs):
    """Send chat completion with exponential backoff retry."""
    last_exception = None
    for attempt in range(max_retries + 1):
        try:
            return client.chat.completions.create(messages=messages, **kwargs)
        except RateLimitError as e:
            last_exception = e
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"[RETRY {attempt+1}/{max_retries}] Rate limited. Waiting {delay:.1f}s...")
                time_mod.sleep(delay)
        except (APIConnectionError, APITimeoutError) as e:
            last_exception = e
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"[RETRY {attempt+1}/{max_retries}] Connection/timeout. Waiting {delay:.1f}s...")
                time_mod.sleep(delay)
        except APIError as e:
            print(f"[ERROR] Non-retryable: {e}")
            raise
    raise RuntimeError(f"Max retries exceeded. Last error: {last_exception}")

print("robust_chat_completion() defined.")
print("Usage: robust_chat_completion(client, messages=[{'role':'user','content':'Hello!'}], model='Qwen/Qwen2.5-0.5B', max_tokens=128)")

In [ ]:
# =============================================
# Cell 24: Server-Side Graceful Shutdown Manager
# =============================================
# NOTE: GPU REQUIRED -- reference for production deployment

import signal, sys, atexit, subprocess as subproc

class VLLMServerManager:
    """Production vLLM server manager with graceful shutdown."""
    def __init__(self):
        self.server_process = None
        signal.signal(signal.SIGTERM, self._handle_shutdown)
        signal.signal(signal.SIGINT, self._handle_shutdown)
        atexit.register(self.cleanup)
    def _handle_shutdown(self, signum, frame):
        print(f"\n[SHUTDOWN] Signal {signum}. Draining requests...")
        self.cleanup()
        sys.exit(0)
    def start(self, model, port=8000, **kwargs):
        cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
               "--model", model, "--port", str(port)]
        for k, v in kwargs.items():
            cmd.append(f"--{k.replace('_', '-')}")
            if v is not True: cmd.append(str(v))
        print(f"[START] {' '.join(cmd)}")
        self.server_process = subproc.Popen(cmd, stdout=subproc.PIPE, stderr=subproc.STDOUT, text=True)
        return self.server_process
    def wait_for_ready(self, timeout=300):
        import httpx
        start = time_mod.time()
        while time_mod.time() - start < timeout:
            try:
                if httpx.get("http://localhost:8000/health", timeout=2.0).status_code == 200:
                    print("[READY] Server accepting requests.")
                    return True
            except Exception: pass
            time_mod.sleep(1)
        print("[TIMEOUT] Server not ready."); return False
    def cleanup(self):
        if self.server_process:
            print("[CLEANUP] Sending SIGTERM...")
            self.server_process.terminate()
            try: self.server_process.wait(timeout=30); print("[CLEANUP] Stopped gracefully.")
            except subproc.TimeoutExpired:
                print("[CLEANUP] Force killing..."); self.server_process.kill(); self.server_process.wait()
        print("[CLEANUP] Done.")

print("VLLMServerManager defined.")
print("Usage: manager = VLLMServerManager(); manager.start('Qwen/Qwen2.5-0.5B'); manager.wait_for_ready(); ...; manager.cleanup()")

## 10. Key Takeaways

1. **vLLM achieves 14-24x higher throughput** than vanilla Hugging Face Transformers through PagedAttention and continuous batching.

2. **OpenAI-compatible API** means zero code changes for existing applications -- just swap `base_url`.

3. **Three key tuning knobs:** `max-num-batched-tokens` (GPU utilization per forward pass), `max-num-seqs` (concurrency limit), and `gpu-memory-utilization` (KV cache budget).

4. **Higher max-num-seqs = higher throughput, higher latency.** Tune based on your SLA: throughput-optimized (128-256), latency-sensitive (16-32), balanced (64).

5. **Continuous batching** eliminates idle GPU time by dynamically adding/removing requests from the batch.

6. **Prometheus metrics** (`/metrics`) give deep visibility into request counts, TTFT, TPOT, and KV cache utilization.

7. **For production, use multiple vLLM instances** with a routing layer (LiteLLM) for multi-model serving.

8. **Always implement graceful shutdown** with request draining in production.

### What's Next

The next notebook covers **SGLang** -- an alternative serving framework with structured generation, RadixAttention for prefix caching, and speculative decoding.